# Data Pre-processing

In [1]:
import sklearn as sk
import pandas as pd

## Import Data

In [2]:
df = pd.read_csv("US_Accidents_March23.csv")
len(df)

7728394

## Basic Data Preprocessing
To simplify your project a bit, use only the data instances that contain at most 1 missing value. According to our counts:
There are 3,554,549 data instances with no missing values:
There are 244,702 data instances with exactly one missing value.
Hence, your dataset for the project should contain 3,554,549 + 244,702 = 3,799,251

In [3]:
df.dropna(thresh=len(df.columns) - 1, inplace=True)
len(df)

3799251

In [4]:
df.drop(columns=['ID', 'Description', 'Country', 'Source', 'Turning_Loop'], inplace = True)
print(df.columns)

Index(['Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng',
       'End_Lat', 'End_Lng', 'Distance(mi)', 'Street', 'City', 'County',
       'State', 'Zipcode', 'Timezone', 'Airport_Code', 'Weather_Timestamp',
       'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)',
       'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)',
       'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing',
       'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station',
       'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Sunrise_Sunset',
       'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight'],
      dtype='str')


In [5]:
basic_df = df.copy(deep = True)

# Handle datetime -> integer by extracting integer values
basic_df["Start_Time"] = pd.to_datetime(basic_df["Start_Time"], format="ISO8601")
basic_df["End_Time"] = pd.to_datetime(basic_df["End_Time"], format="ISO8601")

basic_df["Duration_Minutes"] = (
    basic_df["End_Time"] - basic_df["Start_Time"]
).dt.total_seconds() / 60.0
basic_df["Start_Hour"] = basic_df["Start_Time"].dt.hour
basic_df["Start_DayOfWeek"] = basic_df["Start_Time"].dt.dayofweek  # 0 = Monday, 6 = Sunday
basic_df["Start_Month"] = basic_df["Start_Time"].dt.month
basic_df.drop(columns=["Start_Time", "End_Time"], inplace=True)

In [6]:
# Drop location cols other than lat/long
basic_df.drop(
    columns=[
        "City", "County", "Zipcode", "Airport_Code", "State"
    ],
    inplace=True,
)

In [7]:
#Convert sunset measures
sunset_fields = ["Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]
for f in sunset_fields:
    basic_df[f] = basic_df[f].map({"Day": 1, "Night": 0}) # day = 1, night = 0

In [8]:
# # One-hot encode wind direction and weather condition as dense columns
# weather_columns = ["Wind_Direction", "Weather_Condition"]
# basic_df = pd.get_dummies(
#     basic_df,
#     columns=weather_columns,
#     dummy_na=True,
# )

In [9]:
#Drop timezone, weather
basic_df.drop(columns = ["Timezone", "Weather_Timestamp"], inplace = True)

In [10]:
# Fill missing numeric values one column at a time to avoid allocating another full-size frame.
for column in basic_df.columns:
    if (column == "Street" or column == "Weather_Condition" or column == "Wind_Direction"):
        continue
    
    missing = basic_df[column].isna()
    if missing.any():
        fill_value = basic_df[column].median()
        if pd.isna(fill_value):
            fill_value = 0
        basic_df[column] = basic_df[column].fillna(fill_value)

In [11]:
basic_df.info()

<class 'pandas.DataFrame'>
Index: 3799251 entries, 3402762 to 7728393
Data columns (total 36 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Severity               int64  
 1   Start_Lat              float64
 2   Start_Lng              float64
 3   End_Lat                float64
 4   End_Lng                float64
 5   Distance(mi)           float64
 6   Street                 str    
 7   Temperature(F)         float64
 8   Wind_Chill(F)          float64
 9   Humidity(%)            float64
 10  Pressure(in)           float64
 11  Visibility(mi)         float64
 12  Wind_Direction         str    
 13  Wind_Speed(mph)        float64
 14  Precipitation(in)      float64
 15  Weather_Condition      str    
 16  Amenity                bool   
 17  Bump                   bool   
 18  Crossing               bool   
 19  Give_Way               bool   
 20  Junction               bool   
 21  No_Exit                bool   
 22  Railway                bool 

In [12]:
basic_df.head()

,Severity,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Street,Temperature(F),Wind_Chill(F),Humidity(%),...,Traffic_Calming,Traffic_Signal,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight,Duration_Minutes,Start_Hour,Start_DayOfWeek,Start_Month
3402762,3,40.108910,-83.092860,40.112060,-83.031870,3.230,Outerbelt E,42.1,36.1,58.0,...,False,False,0,0,0,0,360.0,0,0,2
3402767,3,39.172393,-84.492792,39.170476,-84.501798,0.500,I-75 S,37.0,29.8,93.0,...,False,False,1,1,1,1,360.0,7,0,2
3402768,2,39.063240,-84.032430,39.067310,-84.058510,1.427,State Route 32,35.6,29.2,100.0,...,False,True,1,1,1,1,360.0,8,0,2
3402769,2,39.067080,-84.058550,39.063020,-84.032540,1.423,State Route 32,35.6,29.2,100.0,...,False,False,1,1,1,1,360.0,8,0,2
3402771,2,41.375310,-81.820170,41.367860,-81.821740,0.521,I-71 S,33.1,30.0,92.0,...,False,False,1,1,1,1,360.0,11,0,2


## Advanced Preprocessing

In [13]:
import numpy as np


def add_road_type(f):
    hwy_re = (
        r'\b[A-Z]{1,2}-\d+'
        r'|\b(?:Hwy|Fwy|Expy|Pkwy|Tpke|Turnpike|Highway|Freeway|Expressway|Interstate|Route|Rte|SR)\b'
    )
    local_re = (
        r'\b(?:St|Street|Rd|Road|Ave|Avenue|Blvd|Boulevard|Ln|Lane|Dr|Drive|'
        r'Ct|Court|Pl|Place|Way|Pike|Cir|Circle|Trl|Trail|Ter|Terrace)\b'
    )

    street = f['Street']
    f['road_type'] = np.select(
        [
            street.isna(),
            street.str.contains(hwy_re, regex=True, na=False),
            street.str.contains(local_re, regex=True, na=False),
        ],
        [None, 'highway', 'local_street'],
        default='other',
    )
    return f


def weather_type(cond):
    if pd.isna(cond):
        return np.nan
    if 'Fair' in cond or cond == 'Clear':
        return 'clear'
    if 'Thunder' in cond or 'T-Storm' in cond:
        return 'storm'
    if any(term in cond for term in ('Rain', 'Drizzle', 'Precipitation', 'Showers')):
        return 'rain'
    if cond == 'Wintry Mix' or 'Snow' in cond:
        return 'snow'
    if 'Cloudy' in cond or cond == 'Cloudy' or 'Clouds' in cond or 'Overcast' in cond:
        return 'cloudy'
    if any(term in cond for term in ('Fog', 'Mist', 'Smoke', 'Haze', 'Dust', 'Sand')):
        return 'low-visibility'
    if any(term in cond for term in ('Sleet', 'Hail', 'Ice Pellets', 'Tornado')):
        return 'storm'
    if 'Windy' in cond or 'Squall' in cond:
        return 'windy'
    return 'cloudy'


def add_rush_hour(hour):
    return int((7 <= hour <= 9) or (16 <= hour <= 19))


def add_wind_direction_features(F):
    direction_degrees = {
        'N': 0.0,
        'NNE': 22.5,
        'NE': 45.0,
        'ENE': 67.5,
        'E': 90.0,
        'ESE': 112.5,
        'SE': 135.0,
        'SSE': 157.5,
        'S': 180.0,
        'SSW': 202.5,
        'SW': 225.0,
        'WSW': 247.5,
        'W': 270.0,
        'WNW': 292.5,
        'NW': 315.0,
        'NNW': 337.5,
        'NORTH': 0.0,
        'EAST': 90.0,
        'SOUTH': 180.0,
        'WEST': 270.0,
    }

    direction = F['Wind_Direction'].astype('string').str.strip().str.upper()
    degrees = direction.map(direction_degrees).astype('float64')
    radians = np.deg2rad(degrees)
    calm = direction.eq('CALM').fillna(False)
    variable = direction.isin(['VAR', 'VARIABLE', 'VRB']).fillna(False)

    F['Wind_Direction_sin'] = np.sin(radians).fillna(0.0)
    F['Wind_Direction_cos'] = np.cos(radians).fillna(0.0)
    F['Wind_Direction_calm'] = calm.astype('boolean')
    F['Wind_Direction_variable'] = variable.astype('boolean')

    return F.drop(columns=['Wind_Direction'])

### Grouped Features

In [14]:
def add_grouped_features(F):
    poi_type_groups = {
        'traffic_controlled': ['Traffic_Signal', 'Stop', 'Give_Way'],
        'junctions': ['Junction', 'Roundabout', 'No_Exit'],
        'traffic_calming': ['Bump', 'Traffic_Calming'],
        'pedestrians': ['Crossing', 'Station', 'Amenity'],
    }

    for name, cols in poi_type_groups.items():
        F[name] = F[cols].fillna(False).astype(bool).any(axis=1).astype(int)
    F = F.drop(columns=[col for cols in poi_type_groups.values() for col in cols])

    F = add_road_type(F)
    F = F.drop(columns=['Street'])
    F['tod_lighting'] = np.select(
        [F['Sunrise_Sunset'] == 'Day', F['Civil_Twilight'] == 'Day'],
        ['day', 'twilight'],
        default='night',
    )
    return F.drop(
        columns=['Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight']
    )

### Weather and Time Features

In [15]:
def add_weather_and_time_features(F):
    F['Weather_Condition'] = F['Weather_Condition'].apply(weather_type)

    F['Start_Time'] = pd.to_datetime(F['Start_Time'], format='ISO8601')
    F['End_Time'] = pd.to_datetime(F['End_Time'], format='ISO8601')
    F['Duration_Minutes'] = (F['End_Time'] - F['Start_Time']).dt.total_seconds() / 60.0
    F['Start_Hour'] = F['Start_Time'].dt.hour
    F['rush_hour'] = F['Start_Hour'].apply(add_rush_hour)
    F['Start_DayOfWeek'] = F['Start_Time'].dt.dayofweek
    F['Start_Month'] = F['Start_Time'].dt.month
    return F.drop(columns=['End_Time']) # Start time

### Numeric Encoding and Cleanup

In [16]:
def encode_and_clean_advanced(F):
    F = F.drop(
        columns=[
            'City', 'County', 'Zipcode', 'Airport_Code', 'State',
            'End_Lat', 'End_Lng', 'Timezone', 'Weather_Timestamp',
        ]
    )
    F = add_wind_direction_features(F)

    category_codes = {
        'Weather_Condition': {
            'clear': 0,
            'cloudy': 1,
            'rain': 2,
            'snow': 3,
            'storm': 4,
            'low-visibility': 5,
            'windy': 6,
        },
        'road_type': {
            'highway': 0,
            'local_street': 1,
            'other': 2,
        },
        'tod_lighting': {
            'day': 0,
            'twilight': 1,
            'night': 2,
        },
    }
    for column, mapping in category_codes.items():
        F[column] = F[column].map(mapping).fillna(-1).astype('int8')

    numeric_columns = F.select_dtypes(include=['number', 'bool']).columns
    for column in numeric_columns:
        missing = F[column].isna()
        if missing.any():
            fill_value = F[column].median()
            if pd.isna(fill_value):
                fill_value = 0
            F[column] = F[column].fillna(fill_value)

    return F

### Build Advanced Dataframe

In [18]:
def advanced_preprocessing(fold):
    F = fold.copy()
    F = add_grouped_features(F)
    F = add_weather_and_time_features(F)
    return encode_and_clean_advanced(F)


# advanced_processed_data = advanced_preprocessing(df)
# assert advanced_processed_data.select_dtypes(exclude=['number', 'bool']).empty
# advanced_processed_data.info()

### Check dataframe

In [19]:
advanced_processed_data.info()

<class 'pandas.DataFrame'>
Index: 3799251 entries, 3402762 to 7728393
Data columns (total 29 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   Severity                 int64         
 1   Start_Time               datetime64[ns]
 2   Start_Lat                float64       
 3   Start_Lng                float64       
 4   Distance(mi)             float64       
 5   Temperature(F)           float64       
 6   Wind_Chill(F)            float64       
 7   Humidity(%)              float64       
 8   Pressure(in)             float64       
 9   Visibility(mi)           float64       
 10  Wind_Speed(mph)          float64       
 11  Precipitation(in)        float64       
 12  Weather_Condition        int8          
 13  Railway                  bool          
 14  traffic_controlled       int64         
 15  junctions                int64         
 16  traffic_calming          int64         
 17  pedestrians              int64       

In [20]:
advanced_processed_data.head()

,Severity,Start_Time,Start_Lat,Start_Lng,Distance(mi),Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),...,tod_lighting,Duration_Minutes,Start_Hour,rush_hour,Start_DayOfWeek,Start_Month,Wind_Direction_sin,Wind_Direction_cos,Wind_Direction_calm,Wind_Direction_variable
3402762,3,2016-02-08 00:37:08,40.108910,-83.092860,3.230,42.1,36.1,58.0,29.76,10.0,...,2,360.0,0,0,0,2,-0.707107,-0.707107,False,False
3402767,3,2016-02-08 07:53:43,39.172393,-84.492792,0.500,37.0,29.8,93.0,29.69,10.0,...,0,360.0,7,1,0,2,-0.923880,-0.382683,False,False
3402768,2,2016-02-08 08:16:57,39.063240,-84.032430,1.427,35.6,29.2,100.0,29.66,10.0,...,0,360.0,8,1,0,2,-0.923880,-0.382683,False,False
3402769,2,2016-02-08 08:16:57,39.067080,-84.058550,1.423,35.6,29.2,100.0,29.66,10.0,...,0,360.0,8,1,0,2,-0.923880,-0.382683,False,False
3402771,2,2016-02-08 11:51:46,41.375310,-81.820170,0.521,33.1,30.0,92.0,29.63,0.5,...,0,360.0,11,0,0,2,-0.707107,-0.707107,False,False


## Feature Selection with RFE
Select features using a small stratified sample, then apply the selected columns to the full advanced dataset.

In [22]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

target_column = "Severity"
feature_columns = [
    column for column in advanced_processed_data.columns
    if column != target_column
]

sample_size = min(5_000, len(advanced_processed_data))

if sample_size == len(advanced_processed_data):
    rfe_sample = advanced_processed_data.copy()
else:
    rfe_sample, _ = train_test_split(
        advanced_processed_data,
        train_size=sample_size,
        stratify=advanced_processed_data[target_column],
        random_state=42,
    )

X_sample = rfe_sample[feature_columns].to_numpy(
    dtype=np.float32,
    na_value=np.nan,
)
X_sample = np.nan_to_num(
    X_sample,
    nan=0.0,
    posinf=0.0,
    neginf=0.0,
)
y_sample = rfe_sample[target_column]
X_sample = StandardScaler().fit_transform(X_sample)



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\roast\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold

selector = RFECV(
    estimator=LogisticRegression(
        solver="lbfgs",
        max_iter=300,
        class_weight="balanced",
    ),
    step=0.1,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="balanced_accuracy",
    min_features_to_select=5,
    n_jobs=-1,
)

selector.fit(X_sample, y_sample)

selected_features = [
    column
    for column, selected in zip(feature_columns, selector.support_)
    if selected
]

advanced_feature_selected_data = advanced_processed_data[
    selected_features + [target_column]
]

In [ ]:
advanced_feature_selected_data.info()

# Export

In [ ]:
%pip install pyarrow

basic_df.to_parquet(
    "basic_processed_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True
)


advanced_processed_data.to_parquet(
    "advanced_processed_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True
)

advanced_feature_selected_data.to_parquet(
    "advanced_feature_selected_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True,
)